In [0]:
from databricks.sdk import WorkspaceClient
import requests
import json

w = WorkspaceClient()

app_name = "databrck-weather-vector-search"

# 1. Get the OAuth client ID for this Databricks App
app_client_id = w.apps.get(app_name).oauth2_app_client_id

# 2. Get the notebook's current internal Databricks token
notebook_token = (
    dbutils.notebook.entry_point
    .getDbutils()
    .notebook()
    .getContext()
    .apiToken()
    .get()
)

# 3. Exchange it for a token scoped specifically to this app
token_response = requests.post(
    f"{w.config.host}/oidc/v1/token",
    data={
        "grant_type": "urn:ietf:params:oauth:grant-type:token-exchange",
        "subject_token": notebook_token,
        "subject_token_type":
            "urn:databricks:params:oauth:token-type:personal-access-token",
        "requested_token_type":
            "urn:ietf:params:oauth:token-type:access_token",
        "scope": "all-apis",
        "audience": app_client_id,
    },
)

token_response.raise_for_status()
app_token = token_response.json()["access_token"]

# 4. Call your Flask API
# Get the app URL dynamically instead of hardcoding
app_info = w.apps.get(app_name)
url = f"{app_info.url}/api/sync"

payload = {
    "locations": ["Glasscock, TX"],
    "limit": 50
}

response = requests.post(
    url,
    json=payload,
    headers={
        "Authorization": f"Bearer {app_token}",
        "Content-Type": "application/json",
    },
)

print("Status:", response.status_code)
print(json.dumps(response.json(), indent=2))

In [0]:
from weather_client import fetch_alerts
lat = 41.8781
lon = -87.6298

alerts_response = fetch_alerts(lat, lon)

print(
    "Active NWS alerts:",
    len(alerts_response.get("features", []))
)


Finding places which might have weather alerts as Chicago and Austin doesn't have alerts

In [0]:
from weather_client import fetch_alerts

lat = 31.778333333333332
lon = -101.84666666666668
# "Glasscock, TX": {"lat": 31.778333333333332, "lon": -101.84666666666668},
alerts_response = fetch_alerts(lat, lon)

print(
    "Active NWS alerts:",
    len(alerts_response.get("features", []))
)

for feature in alerts_response.get("features", [])[:3]:
    props = feature.get("properties", {})
    print("Event:", props.get("event"))
    print("Headline:", props.get("headline"))
    print("Description:", (props.get("description") or "")[:200])
    print("-" * 80)

In [0]:
import requests

NWS_BASE_URL = "https://api.weather.gov"

HEADERS = {
    "User-Agent": "databricks-weather-vector-search/1.0",
    "Accept": "application/geo+json"
}


def get_candidate_point(geometry):
    """
    Get an approximate center point from a GeoJSON Polygon or MultiPolygon.
    Returns (lat, lon).
    """

    if not geometry:
        return None

    geometry_type = geometry.get("type")
    coordinates = geometry.get("coordinates")

    if not coordinates:
        return None

    if geometry_type == "Polygon":
        # First polygon's exterior ring
        points = coordinates[0]

    elif geometry_type == "MultiPolygon":
        # First polygon's exterior ring
        points = coordinates[0][0]

    else:
        return None

    if not points:
        return None

    # GeoJSON coordinates are [longitude, latitude]
    avg_lon = sum(point[0] for point in points) / len(points)
    avg_lat = sum(point[1] for point in points) / len(points)

    return avg_lat, avg_lon


# ---------------------------------------------------------
# 1. Get ALL currently active NWS alerts
# ---------------------------------------------------------

response = requests.get(
    f"{NWS_BASE_URL}/alerts/active",
    headers=HEADERS,
    timeout=30
)

response.raise_for_status()

data = response.json()
features = data.get("features", [])

print(f"Total active NWS alerts: {len(features)}")


# ---------------------------------------------------------
# 2. Find one whose coordinates work with ?point=
# ---------------------------------------------------------

found = False

for feature in features:

    props = feature.get("properties", {})
    geometry = feature.get("geometry")

    point = get_candidate_point(geometry)

    # Some NWS alerts don't provide polygon geometry
    if point is None:
        continue

    lat, lon = point

    # Verify using exactly the endpoint our project uses
    verify_response = requests.get(
        f"{NWS_BASE_URL}/alerts/active",
        params={"point": f"{lat},{lon}"},
        headers=HEADERS,
        timeout=30
    )

    if verify_response.status_code != 200:
        continue

    point_alerts = verify_response.json().get("features", [])

    if point_alerts:

        print("\n=== ACTIVE ALERT LOCATION FOUND ===")
        print(f"Latitude : {lat}")
        print(f"Longitude: {lon}")
        print(f"Event    : {props.get('event')}")
        print(f"Area     : {props.get('areaDesc')}")
        print(f"Headline : {props.get('headline')}")
        print(f"Alerts returned for this point: {len(point_alerts)}")

        found = True
        break


if not found:
    print(
        "\nActive alerts exist, but I couldn't find one "
        "with usable polygon coordinates."
    )